# Buck age from NDA images: DINOv3 and YOLO

Trains, validates and tests a model that predicts a buck's age class from one image.

**Data.** Only `*_NDA.png` files under `trail cam/images/squared`. The National Deer
Association's ages are the ground truth.

**How the data is used**

| Stage | Data it sees | What it decides |
|---|---|---|
| 1. Select | Development pool, cross-validated | Which architecture to use |
| 2. Train and validate | Development pool, cross-validated | The final models, and an honest out-of-fold score |
| 3. Test | Locked test set, **read once** | Nothing. It only reports the result |

**How leakage is prevented**

- Images of the same deer are grouped, and a group never crosses a split.
- The test set is fixed by `holdout_test_v2.json`, and the file's checksums are verified.
- Augmentation exists only in the training dataset. Validation and test images are untouched.
- Validation and test sets keep their true class mix. Nothing is oversampled.
- Epochs are fixed, so validation scores never pick a checkpoint.
- Architectures are chosen on development data only, and only the winner reaches the test set.

Training itself runs through `buck.benchmark`, so these rules are enforced by code that
raises an error if one is broken.

**To run:** use the `buck-env` kernel and Run All. Finished runs are skipped, so an
interrupted notebook can simply be run again. The test set stays unread until you set
`SCORE_LOCKED_TEST = True` in the next cell.

In [ ]:
# ---- Settings -------------------------------------------------------------
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "buck").is_dir())
IMAGE_ROOT = ROOT / "trail cam" / "images" / "squared"
MANIFEST = ROOT / "trail cam" / "splits" / "holdout_test_v2.json"
METADATA = ROOT / "trail cam" / "image_metadata.csv"
RUNS = ROOT / "benchmark_runs" / "nda_dino_yolo"

# Architectures competing to be the final model.
CANDIDATES = [
    "convnext_small_dinov3",
    "vit_small_dinov3",
    "vit_base_dinov3",
    "cs3darknet_l",
    "yolo11s_cls",
    "yolo26s_cls",
]
# Best known model from earlier work. Trained for comparison, never eligible to win.
REFERENCE = "convnextv2_tiny"

# Stage 1 uses 5 folds to keep the comparison affordable. Stage 2 uses 10, which
# measured best on this corpus. Three seeds because single-seed rankings here have
# failed replication twice.
SELECTION_FOLDS, SELECTION_SEEDS = 5, (42, 43, 44)
FINAL_FOLDS, FINAL_SEEDS = 10, (42, 43, 44)

EPOCHS = 60
AUGMENTATION = "medium"     # light, medium or heavy
TRAIN_MULTIPLIER = 8        # augmented views per training image per epoch
SELECT_METRIC = "qwk"       # ordinal agreement, more stable than accuracy on small folds
SPLIT_SEED = 1337           # must stay 1337 to reproduce the project's folds
MIN_CLASS_COUNT = 8

# Set to a model name to skip the automatic choice in stage 1.
FORCE_WINNER = None

# Flip this only after stage 2, once you are committed to the model. See stage 3.
SCORE_LOCKED_TEST = True

In [ ]:
import csv
import json
import sys
from collections import Counter
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedGroupKFold

sys.path.insert(0, str(ROOT / "src"))

from buck.benchmark import architectures as arch
from buck.benchmark import compare_architectures
from buck.benchmark.data import (
    DEFAULT_EMBEDDING_CACHE,
    MAX_AGE,
    EMBEDDING_MERGE_SIMILARITY,
    EvalDataset,
    TrainDataset,
    _content_digest,
    _embed,
    assert_no_leakage,
    build_groups,
    decode_images,
    drop_rare_classes,
    load_or_create_holdout,
    load_records,
)
from buck.benchmark.ensemble import _probabilities
from buck.benchmark.metrics import bootstrap_ci, confusion, ordinal_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} on {torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU'}")

SURFACE, INK, MUTED, BLUE = "#fcfcfb", "#0b0b0b", "#52514e", "#2a78d6"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": MUTED, "axes.edgecolor": "#d9d8d3",
    "xtick.color": MUTED, "ytick.color": MUTED, "axes.spines.top": False,
    "axes.spines.right": False, "font.size": 11,
})

## Load the NDA images

The age comes from the filename. Only part of the corpus has a row in
`image_metadata.csv`, so the check below compares the filename ages against the
`Correct` column wherever a row exists. Ages above 5.5 share one 5.5+ class.

In [ ]:
records = drop_rare_classes(load_records(IMAGE_ROOT, ("NDA",), ("color", "grayscale")), MIN_CLASS_COUNT)
assert records and all(r.source == "NDA" and r.filename.endswith("_NDA.png") for r in records)

class_ages = sorted({r.age for r in records})
labels = np.array([class_ages.index(r.age) for r in records])
class_names = [f"{a:g}+" if a == MAX_AGE else f"{a:g}" for a in class_ages]

print(f"{len(records)} images in {len(class_ages)} age classes")
print("class mix:", dict(zip(class_names, np.bincount(labels).tolist())))
print("capture:  ", dict(Counter(r.channel for r in records)))

# Filename age against the Correct column. One row describes one deer, but a
# collection batch can hold several deer, so a row only has to match one of them.
correct = {}
with open(METADATA, newline="", encoding="utf-8-sig") as handle:
    for row in csv.DictReader(handle):
        if row["Collected"].strip() and (row.get("Correct") or "").strip():
            correct[row["Collected"].strip()] = min(float(row["Correct"]), MAX_AGE)

batches = {}
for r in records:
    batches.setdefault(r.collected, []).append(r)

matched = sum(any(r.age == age for r in batches.get(week, [])) for week, age in correct.items() if week in batches)
conflicts = [week for week, age in correct.items() if week in batches and not any(r.age == age for r in batches[week])]
covered = sum(r.age == correct[r.collected] for r in records if r.collected in correct)

print(f"\nimages confirmed by the Correct column: {covered} of {len(records)}")
print(f"images labelled by filename alone:      {len(records) - covered}")
print(f"CSV rows with no matching image age:    {len(conflicts)} {conflicts[:5]}")
assert not conflicts, "a Correct value disagrees with every image in its batch; fix it before training"

## Split: locked test set and folds

The test images come from the manifest, never from a fresh random split. Everything
else is the development pool, including new weekly images. This cell repeats the
harness's split exactly, and each run is checked against it later.

In [ ]:
assert MANIFEST.exists(), f"{MANIFEST} is missing; refusing to create a new test set silently"

groups = build_groups(records, verbose=False)
test_mask = load_or_create_holdout(records, groups, MANIFEST, seed=SPLIT_SEED)
test_idx = np.flatnonzero(test_mask)
dev_idx = np.flatnonzero(~test_mask)

# A new image can turn out to be another frame of a locked test deer. It can't
# join the test set or stay in development, so it is left out.
test_groups = set(groups[test_idx].tolist())
collides = np.array([g in test_groups for g in groups[dev_idx]], dtype=bool)
if collides.any():
    print(f"dropping {collides.sum()} development image(s) showing a locked test deer")
dev_idx = dev_idx[~collides]


def make_folds(n_folds):
    splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=SPLIT_SEED)
    return list(splitter.split(np.zeros(len(dev_idx)), labels[dev_idx], groups[dev_idx]))


print(f"development {len(dev_idx)} images in {len(set(groups[dev_idx].tolist()))} animal groups")
print(f"locked test {len(test_idx)} images in {len(test_groups)} animal groups")

## Leakage audit

Every check here stops the notebook if it fails, apart from the last two. Those two
report risks that splitting can't fully remove:

- **Lookalikes across the wall** are image pairs that look almost identical but have
  different ages, so grouping leaves them apart. They may be the same buck in a later
  year.
- **Shared collection batches** mean a test image came in with development images.
  Earlier masking work found that backgrounds alone predict age well above chance, and
  grouping by animal can't detect that kind of leak.

In [ ]:
# 1. No image, and no animal, on both sides of any split, for every fold in use.
for n_folds in sorted({SELECTION_FOLDS, FINAL_FOLDS}):
    for train, val in make_folds(n_folds):
        assert_no_leakage(dev_idx[train], dev_idx[val], test_idx, groups, records)
    print(f"PASS  {n_folds}-fold splits: no image or animal crosses train, validation or test")

# 2. No byte-identical file on both sides, even under another name.
digests = [_content_digest(r.path) for r in records]
shared = {digests[i] for i in dev_idx} & {digests[i] for i in test_idx}
assert not shared, f"{len(shared)} identical file(s) on both sides of the wall"
print("PASS  no identical file in both development and test")

# 3. Evaluation images are deterministic; training images are augmented.
sample_idx = dev_idx[:4]
sample = decode_images([records[i] for i in sample_idx], 224)
evaluation = EvalDataset(sample, labels[sample_idx])
training = TrainDataset(sample, labels[sample_idx], AUGMENTATION, seed=0)
assert torch.equal(evaluation[0][0], evaluation[0][0])
assert any(not torch.equal(training[0][0], training[0][0]) for _ in range(10))
print("PASS  evaluation images are unchanged between reads; training images are augmented")

# 4. Reported, not enforced: lookalikes that grouping leaves apart.
features = _embed(records, DEFAULT_EMBEDDING_CACHE, verbose=False)
similarity = features[test_idx] @ features[dev_idx].T
close = int((similarity > EMBEDDING_MERGE_SIMILARITY).any(axis=1).sum())
print(f"NOTE  {close} of {len(test_idx)} test images have a different-age lookalike in development "
      f"(closest similarity {similarity.max():.3f})")

# 5. Reported, not enforced: test images that share a collection batch with development.
dev_batches = {records[i].collected for i in dev_idx}
same_batch = sum(records[i].collected in dev_batches for i in test_idx)
print(f"NOTE  {same_batch} of {len(test_idx)} test images share a collection batch with development images")

## What augmentation does

Each view below is a random draw from the training pipeline for one development image:
small rotation, horizontal flip, desaturation, brightness, gamma and noise. Geometry is
kept gentle because body proportions are the signal. Test images are never displayed.

In [ ]:
from buck.benchmark.data import augment
import random

preview = decode_images([records[dev_idx[0]]], 224)[0]
rng = random.Random(0)
fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))
axes[0].imshow(preview)
axes[0].set_title("Original", fontsize=10)
for i, ax in enumerate(axes[1:], start=1):
    ax.imshow(augment(preview.copy(), AUGMENTATION, rng))
    ax.set_title(f"Augmented {i}", fontsize=10)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Stage 1: select an architecture

Each candidate is trained with cross-validation on the development pool, once per
seed. The test set isn't read here: `--test-policy none` is hard-coded into every run.

A finished run is reused only if its settings and image counts still match. The folds
change completely when an image is added, so a run from an older corpus can't be
compared and stops the notebook instead.

This stage takes hours. Estimates are in the benchmark README.

In [ ]:
def run_cv(model, n_folds, seed, stage):
    """Train one model with cross-validation, or reuse a finished matching run."""
    out = RUNS / stage / f"{model}_s{seed}"
    results = out / "results.json"
    expected = dict(folds=n_folds, epochs=EPOCHS, augmentation=AUGMENTATION,
                    train_multiplier=TRAIN_MULTIPLIER, seed=seed, split_seed=SPLIT_SEED,
                    checkpoint_policy="final", test_policy="none")

    if results.exists():
        saved = json.loads(results.read_text())
        stale = {k: (saved["config"].get(k), v) for k, v in expected.items() if saved["config"].get(k) != v}
        if saved["n_dev"] != len(dev_idx):
            stale["n_dev"] = (saved["n_dev"], len(dev_idx))
        if saved["n_test"] != len(test_idx):
            stale["n_test"] = (saved["n_test"], len(test_idx))
        if stale:
            raise RuntimeError(f"{out} does not match this notebook (saved, current): {stale}. "
                               f"Delete it or use a new RUNS folder.")
        print(f"reusing {stage}/{out.name}")
        return saved

    compare_architectures.main([
        "--image-root", str(IMAGE_ROOT), "--manifest", str(MANIFEST), "--output", str(out),
        "--sources", "NDA", "--models", model, "--folds", str(n_folds), "--epochs", str(EPOCHS),
        "--augmentation", AUGMENTATION, "--train-multiplier", str(TRAIN_MULTIPLIER),
        "--seed", str(seed), "--split-seed", str(SPLIT_SEED), "--select-metric", SELECT_METRIC,
        "--min-class-count", str(MIN_CLASS_COUNT), "--checkpoint-policy", "final",
        "--test-policy", "none", "--no-profile",
    ])
    return json.loads(results.read_text())


selection = []
for model in [REFERENCE, *CANDIDATES]:
    for seed in SELECTION_SEEDS:
        row = run_cv(model, SELECTION_FOLDS, seed, "selection")["leaderboard"][0]
        selection.append({"model": model, "seed": seed, **row})

In [ ]:
table = pd.DataFrame(selection)
columns = ["cv_accuracy", "cv_within_one", "cv_qwk", "cv_macro_f1", "cv_mae_years", "minutes"]
board = table.groupby("model")[columns].mean()
board["qwk_sd_across_seeds"] = table.groupby("model")["cv_qwk"].std()
board = board.sort_values(f"cv_{SELECT_METRIC}", ascending=False)

# A model near zero agreement failed to train; it has not simply lost.
board["collapsed"] = board["cv_qwk"] < 0.1
eligible = board[board.index.isin(CANDIDATES) & ~board["collapsed"]]
if eligible.empty:
    raise RuntimeError("every candidate collapsed; check learning rates before going further")

WINNER = FORCE_WINNER or eligible.index[0]
display(board.round(3))
print(f"selected: {WINNER}")

# Differences under 0.036 qwk are within seed noise on this corpus.
top = eligible[f"cv_{SELECT_METRIC}"].iloc[0]
tied = [m for m in eligible.index if top - eligible.loc[m, f"cv_{SELECT_METRIC}"] < 0.036]
if SELECT_METRIC == "qwk" and len(tied) > 1:
    print(f"statistically tied with the leader: {tied}. Cost or licensing can break the tie (FORCE_WINNER).")
if board["collapsed"].any():
    print("collapsed, excluded:", list(board.index[board["collapsed"]]))

order = board.index[::-1]
fig, ax = plt.subplots(figsize=(8, 0.5 * len(order) + 1.3))
ax.errorbar(board.loc[order, "cv_qwk"], range(len(order)), xerr=board.loc[order, "qwk_sd_across_seeds"].fillna(0),
            fmt="o", ms=8, color=BLUE, ecolor=BLUE, elinewidth=2, capsize=0)
ax.set_yticks(range(len(order)), [f"{m} (reference)" if m == REFERENCE else m for m in order])
ax.set_xlabel("Cross-validated QWK, mean and SD across seeds (higher is better)")
ax.set_title("Architecture selection on development data", loc="left", fontsize=12)
ax.grid(axis="x", color="#ecebe7", linewidth=1)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## Stage 2: train and validate the final model

The winner is retrained with more folds, and every fold of every seed is kept. The
final model is the average of all those models, which is also what the test set
scores.

**Validation score.** Each development image is predicted only by models that never
trained on it. Because the architecture was chosen on this same pool, this score is
slightly optimistic. The test score is the unbiased one.

In [ ]:
final_runs = {seed: run_cv(WINNER, FINAL_FOLDS, seed, "final") for seed in FINAL_SEEDS}

In [ ]:
size = arch.input_size(WINNER)
mean, std = arch.normalisation(WINNER)


def checkpoints(seed):
    folder = RUNS / "final" / f"{WINNER}_s{seed}" / "checkpoints" / WINNER
    # Numeric order: a plain sort puts fold10 before fold2 and mismatches the folds.
    paths = sorted(folder.glob("fold*.pth"), key=lambda p: int(p.stem[len("fold"):]))
    if len(paths) != FINAL_FOLDS:
        raise RuntimeError(f"expected {FINAL_FOLDS} checkpoints in {folder}, found {len(paths)}")
    return paths


def predict(checkpoint, images, y):
    model = arch.build_model(WINNER, len(class_ages), pretrained=False)
    model.load_state_dict(torch.load(checkpoint, map_location="cpu")["model_state_dict"])
    model.to(device)
    probabilities = _probabilities(model, images, y, 32, device, False, mean, std)
    del model
    torch.cuda.empty_cache()
    return probabilities


def summarise(y, predictions, subset):
    scores = ordinal_metrics(y, predictions, class_ages)
    _, low, high = bootstrap_ci(y, predictions, class_ages, "accuracy")
    return {"subset": subset, "n": scores["n"], "accuracy": scores["accuracy"],
            "accuracy_ci_low": low, "accuracy_ci_high": high, "within_one_class": scores["within_one"],
            "qwk": scores["qwk"], "macro_f1": scores["macro_f1"], "mae_years": scores["mae_years"]}


def by_capture(y, predictions, subset_records, name):
    rows = [summarise(y, predictions, name)]
    channels = np.array([r.channel for r in subset_records])
    for channel, title in (("color", "color only"), ("grayscale", "infrared only")):
        if (channels == channel).any():
            rows.append(summarise(y[channels == channel], predictions[channels == channel], title))
    return pd.DataFrame(rows).set_index("subset")


def plot_confusion(y, predictions, title):
    matrix = np.array(confusion(y, predictions, class_ages))
    ramp = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
        "blue", ["#f0f5fc", "#9ec5f4", "#3987e5", "#184f95"])
    fig, ax = plt.subplots(figsize=(5.2, 4.6))
    ax.imshow(matrix, cmap=ramp)
    ax.set_xticks(range(len(class_names)), class_names)
    ax.set_yticks(range(len(class_names)), class_names)
    ax.set_xlabel("Predicted age (years)")
    ax.set_ylabel("NDA age (years)")
    ax.set_title(title, loc="left", fontsize=12)
    ax.spines[:].set_visible(False)
    for row in range(len(class_names)):
        for col in range(len(class_names)):
            dark = matrix[row, col] > 0.55 * matrix.max()
            ax.text(col, row, matrix[row, col], ha="center", va="center", color="white" if dark else INK)
    plt.tight_layout()
    plt.show()


# Only development images are decoded here. Test pixels are not in memory yet.
dev_records = [records[i] for i in dev_idx]
dev_images = decode_images(dev_records, size)
y_dev = labels[dev_idx]

out_of_fold = np.zeros((len(FINAL_SEEDS), len(dev_idx), len(class_ages)))
for s, seed in enumerate(FINAL_SEEDS):
    for (_, val), checkpoint in zip(make_folds(FINAL_FOLDS), checkpoints(seed)):
        out_of_fold[s, val] = predict(checkpoint, dev_images[val], y_dev[val])

for s, seed in enumerate(FINAL_SEEDS):
    print(f"seed {seed}: accuracy {(out_of_fold[s].argmax(1) == y_dev).mean():.3f}")

validation_predictions = out_of_fold.mean(axis=0).argmax(1)
validation = by_capture(y_dev, validation_predictions, dev_records, "all development")
display(validation.round(3))
plot_confusion(y_dev, validation_predictions, f"{WINNER}: out-of-fold validation")
del dev_images

## Stage 3: test, once

**Read this before setting `SCORE_LOCKED_TEST = True`.** The test score means something
only if nothing is changed because of it. Change the model, settings or data after
seeing it, and it becomes one more validation score. The benchmark README records this
test set as never read, and a read can't be taken back.

Set the switch only when you've accepted the validation result above. If you do go on
to change anything, that needs a new locked test set (`holdout_test_v3.json`).

The report is saved on the first read. Running this cell again shows the saved report
and doesn't score again. If a different model is selected, the notebook stops.

With about 58 test images, expect a 95% interval of roughly ±12 points.

In [ ]:
REPORT = RUNS / "locked_test_report.json"
y_test = labels[test_idx]
test_records = [records[i] for i in test_idx]

if REPORT.exists():
    report = json.loads(REPORT.read_text())
    if report["model"] != WINNER:
        raise RuntimeError(f"the test set was already read for {report['model']} on {report['created']}. "
                           f"Scoring {WINNER} as well would turn it into a selection set.")
    print(f"showing the saved report from {report['created']}; the test set was not read again")
    test_predictions = np.array(report["predictions"])
elif not SCORE_LOCKED_TEST:
    report = test_predictions = None
    print("test set untouched. Set SCORE_LOCKED_TEST = True when you have committed to this model.")
else:
    test_images = decode_images(test_records, size)
    probabilities = [predict(c, test_images, y_test) for seed in FINAL_SEEDS for c in checkpoints(seed)]
    test_predictions = np.mean(probabilities, axis=0).argmax(1)
    report = {"model": WINNER, "created": datetime.now().isoformat(timespec="seconds"),
              "manifest": MANIFEST.name, "seeds": list(FINAL_SEEDS), "folds": FINAL_FOLDS,
              "n_models_averaged": len(probabilities), "n_dev": len(dev_idx), "n_test": len(test_idx),
              "predictions": test_predictions.tolist()}
    REPORT.write_text(json.dumps(report, indent=2))
    del test_images

if test_predictions is not None:
    test_scores = by_capture(y_test, test_predictions, test_records, "all test")
    display(test_scores.round(3))
    plot_confusion(y_test, test_predictions, f"{WINNER}: locked test set")
    comparison = pd.concat([validation.loc[["all development"]], test_scores.loc[["all test"]]])
    display(comparison.round(3))
    print("With this few test images the two can differ by chance. The test score is the one to quote.")